# Hệ thống Tư vấn Chọn Bất động sản Thông minh (Nhóm 8 - IT2041)
## Demo Pipeline 5.1: Rule-based Filtering & Scoring (Midterm)

Notebook này gom toàn bộ các bước xử lý dữ liệu và chạy pipeline đề xuất của dự án để chạy thử nghiệm một cách trực quan, bao gồm:
1. **Tải dữ liệu gốc** và lọc subset BĐS Quận Gò Vấp.
2. **Làm giàu tiện ích (POI Enrichment)**: Tính khoảng cách thực tế đến các điểm tiện ích.
3. **Chạy thuật toán lọc (Filtering)** & **tính điểm (Scoring)** theo 3 kịch bản người dùng khác nhau.
4. **Hiển thị trực quan** kết quả đề xuất Top 5 dưới dạng bảng dữ liệu.

In [ ]:
import os
import csv
import json
import math
import pandas as pd

# Thiết lập hiển thị Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

### Bước 1: Trích xuất và làm sạch dữ liệu BĐS Gò Vấp
Chúng ta sẽ đọc dữ liệu từ file gốc `docs/data_public.csv` (chứa hơn 51,000 listings tại TP.HCM) và lọc ra các tin đăng **Nhà riêng** tại **Quận Gò Vấp** thỏa mãn các tiêu chí sạch (đầy đủ tọa độ, số phòng ngủ/phòng vệ sinh hợp lệ).

In [ ]:
# Đường dẫn tương đối từ thư mục notebooks/
INPUT_CSV = '../docs/data_public.csv'

def is_valid_row(r):
    if r.get('Property Type', '').strip() != 'Nhà riêng':
        return False
    if not r.get('Latitude', '').strip() or not r.get('Longitude', '').strip():
        return False
    if not r.get('Bedrooms', '').strip() or not r.get('Bathrooms', '').strip():
        return False
    try:
        beds = int(r['Bedrooms'])
        baths = int(r['Bathrooms'])
        if beds < 1 or beds > 10 or baths < 1 or baths > 10:
            return False
    except (ValueError, TypeError):
        return False
    try:
        price = float(r['Price'])
        if price < 500 or price > 30000: # lọc giá từ 500tr đến 30 tỷ
            return False
    except (ValueError, TypeError):
        return False
    try:
        area = float(r['Area'])
        if area < 20 or area > 500: # diện tích hợp lý từ 20m2 đến 500m2
            return False
    except (ValueError, TypeError):
        return False
    return True

def is_go_vap(r):
    text = (r.get('Location', '') + ' ' + r.get('Title', '')[:200]).lower()
    return 'gò vấp' in text or 'go vap' in text

def clean_row(r, idx):
    price_million = float(r['Price'])
    area_m2 = float(r['Area'])
    location = r.get('Location', '')
    
    # Tách Phường
    ward = ''
    parts = location.split(',')
    for p in parts:
        p = p.strip()
        if 'Phường' in p or 'phường' in p:
            ward = p
            break
            
    return {
        'property_id': f'GV_{idx:03d}',
        'title': r.get('Title', '').strip()[:80],
        'district': 'Gò Vấp',
        'ward': ward,
        'price_million_vnd': price_million,
        'price_billion_vnd': round(price_million / 1000, 2),
        'area_m2': area_m2,
        'price_per_m2_million': round(price_million / area_m2, 2) if area_m2 > 0 else 0,
        'bedrooms': int(r['Bedrooms']),
        'bathrooms': int(r['Bathrooms']),
        'latitude': float(r['Latitude']),
        'longitude': float(r['Longitude'])
    }

# Đọc và lọc
with open(INPUT_CSV, 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    go_vap_rows = [r for r in reader if is_valid_row(r) and is_go_vap(r)]

properties = [clean_row(r, i + 1) for i, r in enumerate(go_vap_rows)]
df_raw = pd.DataFrame(properties)
print(f"Đã trích xuất được {len(df_raw)} BĐS hợp lệ tại Gò Vấp.")
df_raw.head(3)

### Bước 2: Làm giàu dữ liệu khoảng cách tiện ích (POI Enrichment)
Chúng ta định nghĩa cơ sở dữ liệu các tọa độ địa điểm thực tế tại Quận Gò Vấp và dùng công thức Haversine để tính khoảng cách đường chim bay (mét) đến điểm tiện ích gần nhất cho mỗi BĐS.

In [ ]:
POI_DATABASE = {
    'school': [
        {'name': 'Trường THCS Nguyễn Du', 'lat': 10.8405, 'lon': 106.6550},
        {'name': 'Trường THPT Trần Hưng Đạo', 'lat': 10.8370, 'lon': 106.6635},
        {'name': 'Trường TH Lương Thế Vinh', 'lat': 10.8480, 'lon': 106.6520},
        {'name': 'Trường THCS Gò Vấp', 'lat': 10.8330, 'lon': 106.6450},
    ],
    'park': [
        {'name': 'Công viên Gia Định', 'lat': 10.8190, 'lon': 106.6770},
        {'name': 'Công viên Làng Hoa', 'lat': 10.8430, 'lon': 106.6580},
    ],
    'hospital': [
        {'name': 'Bệnh viện Quận Gò Vấp', 'lat': 10.8380, 'lon': 106.6500},
        {'name': 'Bệnh viện 175', 'lat': 10.8570, 'lon': 106.6640},
    ],
    'supermarket': [
        {'name': 'Emart Gò Vấp', 'lat': 10.8345, 'lon': 106.6575},
        {'name': 'Co.opmart Quang Trung', 'lat': 10.8400, 'lon': 106.6470},
        {'name': 'VinMart Thống Nhất', 'lat': 10.8430, 'lon': 106.6650},
    ],
    'boulevard': [
        {'name': 'Quang Trung (trục chính)', 'lat': 10.8380, 'lon': 106.6450},
        {'name': 'Nguyễn Oanh', 'lat': 10.8420, 'lon': 106.6700},
        {'name': 'Phạm Văn Đồng', 'lat': 10.8200, 'lon': 106.6830},
    ],
}

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

def enrich_property(prop):
    lat, lon = prop['latitude'], prop['longitude']
    enriched = dict(prop)
    for poi_type, pois in POI_DATABASE.items():
        distances = [haversine_m(lat, lon, p['lat'], p['lon']) for p in pois]
        enriched[f'distance_to_nearest_{poi_type}_m'] = round(min(distances))
    return enriched

properties_enriched = [enrich_property(p) for p in properties]
df_enriched = pd.DataFrame(properties_enriched)
df_enriched.head(3)

### Bước 3: Định nghĩa các kịch bản người dùng mẫu (Personas)
Chúng ta mô tả 3 đối tượng người dùng với các ràng buộc cứng (Hard Constraints) và trọng số mong muốn (Soft Preferences).

In [ ]:
USER_SCENARIOS = [
    {
        'scenario_id': 'family_01',
        'name': 'Gia đình có con nhỏ',
        'hard_constraints': {
            'budget_max_million': 8000, # dưới 8 tỷ
            'min_bedrooms': 3,
        },
        'soft_preferences': {
            'price': {'weight': 0.25, 'direction': 'lower_better', 'min': 3000, 'max': 8000},
            'distance_to_nearest_school_m': {'weight': 0.25, 'direction': 'lower_better', 'min': 0, 'max': 2000},
            'distance_to_nearest_park_m': {'weight': 0.20, 'direction': 'lower_better', 'min': 0, 'max': 2000},
            'distance_to_nearest_supermarket_m': {'weight': 0.15, 'direction': 'lower_better', 'min': 0, 'max': 1500},
            'area_m2': {'weight': 0.15, 'direction': 'higher_better', 'min': 30, 'max': 150},
        }
    },
    {
        'scenario_id': 'young_pro_01',
        'name': 'Người trẻ độc thân',
        'hard_constraints': {
            'budget_max_million': 5500,
            'min_bedrooms': 2,
        },
        'soft_preferences': {
            'price': {'weight': 0.35, 'direction': 'lower_better', 'min': 2000, 'max': 5500},
            'distance_to_nearest_supermarket_m': {'weight': 0.25, 'direction': 'lower_better', 'min': 0, 'max': 1500},
            'distance_to_nearest_boulevard_m': {'weight': 0.20, 'direction': 'lower_better', 'min': 0, 'max': 2000},
            'area_m2': {'weight': 0.20, 'direction': 'higher_better', 'min': 20, 'max': 100},
        }
    },
    {
        'scenario_id': 'investor_01',
        'name': 'Nhà đầu tư',
        'hard_constraints': {
            'budget_max_million': 15000,
            'min_bedrooms': 1,
        },
        'soft_preferences': {
            'price_per_m2': {'weight': 0.30, 'direction': 'lower_better', 'min': 30, 'max': 200},
            'distance_to_nearest_boulevard_m': {'weight': 0.25, 'direction': 'lower_better', 'min': 0, 'max': 2000},
            'area_m2': {'weight': 0.25, 'direction': 'higher_better', 'min': 30, 'max': 250},
            'distance_to_nearest_supermarket_m': {'weight': 0.20, 'direction': 'lower_better', 'min': 0, 'max': 1500},
        }
    }
]

### Bước 4: Chạy Pipeline Lọc & Đề xuất Top 5
Hàm dưới đây sẽ tính toán điểm chuẩn hóa cho các tiêu chí ưu tiên của khách hàng, sắp xếp và đưa ra Top 5 khuyến nghị tối ưu nhất.

In [ ]:
def normalize_val(val, limit_min, limit_max, direction):
    if direction == 'lower_better':
        return max(0.0, min(1.0, (limit_max - val) / (limit_max - limit_min)))
    elif direction == 'higher_better':
        return max(0.0, min(1.0, (val - limit_min) / (limit_max - limit_min)))
    return 0.0

def get_attr_value(prop, attr):
    if attr == 'price':
        return prop['price_million_vnd']
    elif attr == 'price_per_m2':
        return prop['price_per_m2_million']
    return prop.get(attr, 0)

def run_recommender(scenario, items):
    hc = scenario['hard_constraints']
    sp = scenario['soft_preferences']
    
    # 1. Hard filtering
    candidates = []
    for item in items:
        if item['price_million_vnd'] <= hc['budget_max_million'] and item['bedrooms'] >= hc['min_bedrooms']:
            candidates.append(item)
            
    # 2. Scoring
    results = []
    for cand in candidates:
        score = 0.0
        for attr, config in sp.items():
            val = get_attr_value(cand, attr)
            norm = normalize_val(val, config['min'], config['max'], config['direction'])
            score += norm * config['weight']
            
        cand_res = dict(cand)
        cand_res['score'] = round(score, 4)
        results.append(cand_res)
        
    # 3. Sort & Top 5
    results.sort(key=lambda x: x['score'], reverse=True)
    return pd.DataFrame(results[:5])

#### Chạy và hiển thị kết quả cho từng Persona

In [ ]:
for scenario in USER_SCENARIOS:
    print(f"\n🏆 TOP 5 ĐỀ XUẤT CHO: {scenario['name'].upper()}")
    df_top5 = run_recommender(scenario, properties_enriched)
    
    # Chọn các cột để show đẹp mắt
    show_cols = ['property_id', 'score', 'price_billion_vnd', 'area_m2', 'bedrooms', 'bathrooms']
    for col in scenario['soft_preferences'].keys():
        # map tên cột hiển thị tiện ích
        actual_col = f'distance_to_nearest_{col.replace("distance_to_nearest_", "")}'
        if actual_col in df_top5.columns:
            show_cols.append(actual_col)
        elif col in df_top5.columns:
            show_cols.append(col)
            
    show_cols.append('title')
    display(df_top5[show_cols])